# AI Policy Gap & Impact Analyzer

## Project Summary
This notebook implements an AI-powered public policy analysis system.
Users can upload multiple government scheme documents (PDFs), which are analyzed using Large Language Models (LLMs) to extract eligibility criteria, benefits, constraints, and target demographics.

The system performs cross-policy reasoning to:
- identify overlaps between schemes
- detect underserved or excluded groups
- highlight policy conflicts and gaps

Finally, it supports follow-up analytical questions through a conversational interface.


In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("groq")


In [ ]:
!pip install PyMuPDF
!pip install langchain-groq
import fitz  # PyMuPDF
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from collections import defaultdict

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"


In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("groq")



In [ ]:
#📄 Upload multiple Government Scheme PDFs for policy analysis

from google.colab import files

uploaded = files.upload()
file_paths = list(uploaded.keys())

print(f"Loaded {len(file_paths)} document(s):")
for path in file_paths:
    print("-", path)



In [ ]:
# 📄 Extract Text from PDFs & Create Policy-Aware Chunks

import fitz  # PyMuPDF
from langchain_core.documents import Document


def extract_text_pymupdf(path: str) -> str:
    """
    Extract raw text from a PDF file using PyMuPDF.
    """
    doc = fitz.open(path)
    return "\n".join(page.get_text("text") for page in doc)


def chunk_raw_text(
    text: str,
    chunk_size: int = 1000,
    overlap: int = 150,
    source: str = "pdf"
):
    """
    Split text into overlapping chunks and attach metadata.
    """
    chunks = []
    cid = 0
    start, n = 0, len(text)

    while start < n:
        end = min(start + chunk_size, n)

        chunks.append(
            Document(
                page_content=text[start:end],
                metadata={
                    "source": source,
                    "chunk_id": cid
                }
            )
        )

        cid += 1
        if end == n:
            break

        start = max(0, end - overlap)

    return chunks


# 🔁 Process ALL uploaded policy PDFs
all_policy_docs = []

for pdf_path in file_paths:  # file_paths comes from upload cell
    raw_text = extract_text_pymupdf(pdf_path).strip()
    print(f"{pdf_path} → characters extracted:", len(raw_text))

    policy_chunks = chunk_raw_text(
        raw_text,
        source=pdf_path
    )

    # Attach policy identity to every chunk
    for doc in policy_chunks:
        doc.metadata["scheme_name"] = pdf_path

    all_policy_docs.extend(policy_chunks)

print("Total chunks across all policy documents:", len(all_policy_docs))


In [ ]:
import os
from google.colab import userdata

groq_key = userdata.get("groq")

print("Groq key found:", groq_key is not None)

if groq_key is None or len(groq_key.strip()) < 10:
    raise ValueError("Groq API key is missing or invalid")

os.environ["GROQ_API_KEY"] = groq_key

print("Groq API key successfully set in environment")


In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)


In [ ]:
print(llm.invoke("Say OK").content)


In [ ]:
# 🧠 Policy Feature Extraction using LLM
# This cell analyzes each government scheme and extracts structured insights

from collections import defaultdict

policy_feature_map = defaultdict(list)

print("Extracting structured policy features...\n")

for doc in all_policy_docs:
    scheme = doc.metadata["scheme_name"]

    prompt = f"""
    You are a public policy analyst.

    Analyze the following excerpt from a government scheme and extract:

    - Primary objective
    - Target beneficiaries
    - Eligibility criteria
    - Benefits provided
    - Key limitations or exclusions

    Return concise bullet points only.
    Do NOT repeat the text verbatim.
    Be factual and neutral.

    Scheme Name: {scheme}

    Scheme Excerpt:
    {doc.page_content}
    """

    response = llm.invoke(prompt).content

    policy_feature_map[scheme].append(response)

print("Policy feature extraction completed.")


In [ ]:
# 🔹 Aggregate Chunk-Level Analysis per Scheme

aggregated_policy_summaries = {}

for scheme, analyses in policy_feature_map.items():
    combined = "\n".join(analyses)

    prompt = f"""
    You are consolidating policy analysis.

    Combine the following extracted insights into a single,
    clean summary for the scheme.

    Remove redundancy.
    Keep only the most important points.

    Extracted Insights:
    {combined}
    """

    aggregated_policy_summaries[scheme] = llm.invoke(prompt).content

print("Policy summaries aggregated per scheme.")


In [ ]:
# 🔥 Cross-Policy Gap & Overlap Analysis (LLM Call #2) — TABULAR OUTPUT

print("Performing cross-policy reasoning...\n")

# Combine aggregated summaries from all schemes
combined_policy_analysis = ""

for scheme, summary in aggregated_policy_summaries.items():
    combined_policy_analysis += f"\n\n### Scheme: {scheme}\n{summary}\n"

# Prompt for higher-level reasoning across policies (TABLE FORMAT)
prompt = f"""
You are a senior public policy expert.

Given the following consolidated analyses of multiple government schemes:

{combined_policy_analysis}

Analyze the policies and produce a TABLE in MARKDOWN format with the following columns:

| Issue Type | Description | Affected Schemes | Affected Demographics | Severity (Low/Medium/High) |

Rules:
- Each row should represent ONE distinct insight.
- Issue Type must be one of:
  - Overlap
  - Exclusion
  - Conflict
  - Gap
- Be concise, analytical, and neutral.
- Do NOT repeat scheme descriptions verbatim.
- Return ONLY the table. No extra text before or after.
"""

cross_policy_insights = llm.invoke(prompt).content

print("Cross-policy analysis completed.\n")
print(cross_policy_insights)


### Cross-Policy Reasoning
The following section performs higher-level reasoning across all analyzed schemes
to identify overlaps, exclusions, conflicts, and policy gaps that are not visible
when examining a single document in isolation.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import io

# Parse the markdown table into a DataFrame
data = io.StringIO(cross_policy_insights)
df = pd.read_csv(data, sep='|', skipinitialspace=True)

# Clean up DataFrame (remove empty first/last columns and strip whitespace from column names)
df = df.iloc[1:]  # Remove the separator line
df = df.dropna(axis=1, how='all')
df.columns = [col.strip() for col in df.columns]

# ---- Step 1: Identify correct severity column dynamically ----

print("Available columns:", df.columns.tolist())

severity_col = None
for col in df.columns:
    if "severity" in col.lower():
        severity_col = col
        break

assert severity_col is not None, "No severity column found in table"

# ---- Step 2: Visualize Issue Types ----

plt.figure()
df["Issue Type"].value_counts().plot(kind="bar")
plt.title("Distribution of Cross-Policy Issues by Type")
plt.xlabel("Issue Type")
plt.ylabel("Number of Issues")
plt.show()

# ---- Step 3: Visualize Severity Levels ----

plt.figure()
df[severity_col].value_counts().plot(kind="bar")
plt.title("Severity Distribution of Policy Issues")
plt.xlabel("Severity Level")
plt.ylabel("Number of Issues")
plt.show()

In [ ]:
# 📝 Plain-Language Insights Summary

summary_prompt = f"""
You are explaining policy insights to an educated non-expert audience.

Based on the following policy analysis, summarize the key findings in
simple, clear language that a general reader can understand.

Avoid technical jargon.
Focus on what matters to citizens.

Policy Analysis:
{cross_policy_insights}
"""

plain_language_summary = llm.invoke(summary_prompt).content

print("Plain-Language Summary:\n")
print(plain_language_summary)


In [ ]:
# 🧭 Policy Recommendations & Improvements

recommendation_prompt = f"""
You are a public policy advisor.

Based on the analysis below, suggest:
1. Policy improvements
2. New policy ideas or adjustments
3. Ways to reduce exclusion or overlap

Keep recommendations realistic and actionable.

Policy Analysis:
{cross_policy_insights}
"""

policy_recommendations = llm.invoke(recommendation_prompt).content

print("Policy Recommendations:\n")
print(policy_recommendations)


In [ ]:
# 💬 Interactive Policy Chatbot with Memory (Robust Version)

print("Policy Chatbot Ready!")
print("Ask questions like:")
print("What is the eligibilty criteria")
print("\nType 'exit' to stop.\n")

# Manual conversation memory
chat_history = []

# Fixed grounding context (from documents)
base_context = f"""
Policy Analysis:
{cross_policy_insights}

Plain Summary:
{plain_language_summary}

Policy Recommendations:
{policy_recommendations}
"""

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    # Build conversation context
    conversation = ""
    for turn in chat_history:
        conversation += f"User: {turn['user']}\nAI: {turn['ai']}\n"

    prompt = f"""
You are a policy analysis assistant.
Answer ONLY using the information provided below.
If the answer is not present, say "I don't know based on the documents."

{base_context}

Conversation so far:
{conversation}

User question:
{user_input}
"""

    ai_response = llm.invoke(prompt).content

    print("AI:", ai_response)

    # Save memory
    chat_history.append({
        "user": user_input,
        "ai": ai_response
    })


## Conclusion

This project demonstrates how Large Language Models can be used for **structured analysis and reasoning across multiple real-world documents**, rather than simple question answering.

By processing multiple government policy PDFs, the system:
- extracts key policy features using an LLM,
- consolidates insights at the scheme level,
- performs cross-policy reasoning to identify overlaps, gaps, and exclusions,
- translates complex analysis into plain-language summaries, and
- enables interactive, memory-aware follow-up conversations grounded in document-derived insights.

The design intentionally avoids heavy infrastructure (such as vector databases or live retrieval) in favor of a **transparent, stable, and interpretable reasoning pipeline**, making the analysis easier to understand, audit, and explain.

This approach highlights the potential of LLMs as **decision-support tools** for public policy analysis, enabling clearer insights for policymakers, researchers, and citizens alike.
